# Sound Generation via ALD-SC (Latent Sound Generation)

**End-to-end audio synthesis** with ArrowSpace Latent Diffusion and Spectral Chart
Conditioning. This notebook runs the full pipeline:

1. Build ArrowSpace prior from audio features
2. Train graph decoder vs baseline decoder
3. Train 1-D DiT denoiser
4. **Sample → decode → play audio** (with interactive knobs)
5. Evaluate reconstruction quality and λ_ED ablation

> **Central claim:** decoding on the feature-space manifold
> $(L_F, \lambda^{ED})$ yields better reconstruction than unconstrained decoding.

Adjust the knobs in the next cell to generate different sounds.

In [1]:
# --- Interactive Knobs ---
SEED = 3407           # random seed for sampling
STEPS = 50            # number of DDIM sampling steps
USE_C_SPEC = True     # enable spectral conditioning (lambda_ED ablation toggle)
TEMPERATURE = 0.75     # noise scaling during sampling (lower = more conservative)
CLIP_INDEX = 0        # which training clip to use for c_spec reference

# --- Training Knobs ---
DECODER_EPOCHS = 20   # decoder training epochs (short for demo)
DIFFUSION_EPOCHS = 20 # DiT training epochs (short for demo)
NUM_SAMPLES = 64      # number of synthetic clips for the corpus
AUDIO_LENGTH = 24000  # 1 second @ 24kHz (short for CPU demo)

print(f'Knobs: seed={SEED}, steps={STEPS}, use_c_spec={USE_C_SPEC}, '
      f'temp={TEMPERATURE}, clip={CLIP_INDEX}')

Knobs: seed=3407, steps=50, use_c_spec=True, temp=0.75, clip=0


In [2]:
import torch
import torch.nn as nn
import torchaudio
from IPython.display import Audio, display

from ald_sc.build_prior import build_arrow_prior
from ald_sc.audio_codec import BaselineAudioDecoder, AudioVAE
from ald_sc.graph_decoder import GraphDecoder
from ald_sc.dit import MinimalDiT
from ald_sc.data import ToyAudioDataset, build_audio_dataloader
from ald_sc.losses import ALDSCLoss
from ald_sc.schedule import CosineSchedule
from ald_sc.sampling import sample_ddim
from ald_sc.trainer import train_audio_decoder, train_audio_diffusion

device = torch.device('cpu')
torch.manual_seed(SEED)
print('Imports done. Device:', device)

Imports done. Device: cpu


## Step 1: Build the ArrowSpace Prior

Extract audio features from the corpus and build the frozen ArrowSpace
prior $(L_F, U_q, \lambda^{ED})$. We use a **stub encoder** for CPU-only
execution; swap to `EnCodecEncoder()` for real audio features.

In [3]:
# Stub encoder: Conv1d with stride 320 (mimics EnCodec 24kHz)
class StubEncoder(nn.Module):
    def __init__(self, latent_dim=128, stride=320):
        super().__init__()
        self.proj = nn.Conv1d(1, latent_dim, stride, stride=stride)

    def encode(self, x, prior):
        z = self.proj(x).float()
        a = z.mean(dim=2)
        c_spec = prior.chart_energy_descriptor(a)
        return z, a, c_spec

    def extract_features(self, x):
        return self.proj(x).float()

encoder = StubEncoder()

# Extract features from corpus
dataset = ToyAudioDataset(num_samples=NUM_SAMPLES, audio_length=AUDIO_LENGTH)
loader = build_audio_dataloader(dataset, batch_size=8, shuffle=False)

features = []
for batch in loader:
    z = encoder.extract_features(batch)
    features.append(z.mean(dim=2))
embeddings = torch.cat(features, dim=0)
print(f'Corpus embeddings: {embeddings.shape}')

# Build prior
prior = build_arrow_prior(embeddings, q=8, k=4)
print(f'L_F: {prior.L_F.shape}, U_q: {prior.U_q.shape}, q={prior.q}')

Corpus embeddings: torch.Size([64, 128])
L_F: torch.Size([128, 128]), U_q: torch.Size([128, 8]), q=8


## Step 2: Train Graph Decoder vs Baseline Decoder

Train two decoders with **matched capacity** -- the only variable is
graph structure (U_q + lambda_ED gating).

In [4]:
# Graph decoder (uses L_F / U_q / lambda_ED)
graph_decoder = GraphDecoder(
    latent_channels=128, out_channels=1, feature_dim=128,
    base_channels=32, prior=prior, upsample_strides=(2, 4, 5, 8),
)

# Baseline decoder (no graph structure, matched capacity)
baseline_decoder = BaselineAudioDecoder(
    latent_channels=128, out_channels=1, base_channels=32,
    upsample_strides=(2, 4, 5, 8),
)

loss_fn = ALDSCLoss(prior=prior, lambda_rec=1.0, lambda_stft=0.0,
                    lambda_chart=0.5, lambda_smooth=0.1)
train_loader = build_audio_dataloader(dataset, batch_size=4, shuffle=True)

# Train graph decoder
graph_vae = AudioVAE(encoder=encoder, decoder=graph_decoder)
print('Training graph decoder...')
graph_losses = list(train_audio_decoder(train_loader, graph_vae, prior, loss_fn,
                                          epochs=DECODER_EPOCHS, lr=1e-3, device=device))
print(f'  Loss: {graph_losses[0]["loss"]:.4f} -> {graph_losses[-1]["loss"]:.4f}')

# Train baseline decoder
baseline_vae = AudioVAE(encoder=encoder, decoder=baseline_decoder)
print('Training baseline decoder...')
baseline_losses = list(train_audio_decoder(train_loader, baseline_vae, prior, loss_fn,
                                           epochs=DECODER_EPOCHS, lr=1e-3, device=device))
print(f'  Loss: {baseline_losses[0]["loss"]:.4f} -> {baseline_losses[-1]["loss"]:.4f}')

Training graph decoder...
  Loss: 0.5460 -> 0.1133
Training baseline decoder...
  Loss: 0.5498 -> 0.1071


## Step 3: Train the 1-D DiT Denoiser

Train the unconditional 1-D DiT on EnCodec latents (time-only AdaLN).

In [5]:
latent_length = AUDIO_LENGTH // 320  # 75 for 24000 samples
dit = MinimalDiT(
    latent_channels=128, latent_length=latent_length,
    patch_size=8, dim=64, depth=2, num_heads=4, spec_dim=24,
)
sched = CosineSchedule(num_steps=1000)

# Freeze VAE for diffusion training
for p in graph_vae.parameters():
    p.requires_grad_(False)

print(f'Training DiT (latent_length={latent_length})...')
diff_losses = list(train_audio_diffusion(train_loader, graph_vae, dit, prior, sched,
                                          epochs=DIFFUSION_EPOCHS, lr=1e-3, device=device))
print(f'  Loss: {diff_losses[0]["loss"]:.4f} -> {diff_losses[-1]["loss"]:.4f}')

Training DiT (latent_length=75)...
  Loss: 0.6536 -> 0.6013


## Step 4: Generate Audio

Sample a latent from noise using DDIM, then decode it into a waveform.
The knobs control the generation:
- **SEED** -- different seeds give different sounds
- **STEPS** -- more steps = higher quality, slower
- **USE_C_SPEC** -- toggle lambda_ED gating (ablation)
- **TEMPERATURE** -- scale the initial noise (lower = quieter/more conservative)

In [6]:
# Sample latent z from noise
torch.manual_seed(SEED)
dit = dit.eval()
z = sample_ddim(dit, sched, batch_size=1, steps=STEPS, seed=SEED, device=device)

# Apply temperature scaling
z = z * TEMPERATURE
print(f'Sampled z: {z.shape}')

# Derive c_spec from z (self-consistent decoding)
a = z.mean(dim=2)
c_spec = prior.chart_energy_descriptor(a)

# Decode with graph decoder
with torch.no_grad():
    if USE_C_SPEC:
        audio_graph = graph_decoder(z, c_spec)
    else:
        # lambda_ED ablation: zero c_spec to disable gating
        audio_graph = graph_decoder(z, torch.zeros_like(c_spec))

    audio_baseline = baseline_decoder(z)

# Normalize for playback
def normalize(audio):
    audio = audio.squeeze(0)
    peak = audio.abs().max()
    if peak > 0:
        audio = audio / peak
    return audio

audio_graph_norm = normalize(audio_graph)
audio_baseline_norm = normalize(audio_baseline)

print(f'Graph decoder audio: {audio_graph_norm.shape}, '
      f'{audio_graph_norm.shape[-1]/24000:.2f}s')
print(f'Baseline decoder audio: {audio_baseline_norm.shape}, '
      f'{audio_baseline_norm.shape[-1]/24000:.2f}s')

Sampled z: torch.Size([1, 128, 75])
Graph decoder audio: torch.Size([1, 24000]), 1.00s
Baseline decoder audio: torch.Size([1, 24000]), 1.00s


In [7]:
# Play graph decoder output
print('Graph decoder output:')
display(Audio(audio_graph_norm.numpy(), rate=24000))

Graph decoder output:


In [8]:
# Play baseline decoder output
print('Baseline decoder output:')
display(Audio(audio_baseline_norm.numpy(), rate=24000))

Baseline decoder output:


In [9]:
# Compare with a real training clip
print('Real training clip (reference):')
real_clip = dataset[CLIP_INDEX]  # (1, T)
display(Audio(real_clip.numpy(), rate=24000))

Real training clip (reference):


## Step 5: Evaluation

Compare reconstruction quality: graph decoder vs baseline decoder.
The graph decoder should have lower reconstruction error if the
central claim holds.

In [10]:
# Reconstruct held-out clips with both decoders
eval_loader = build_audio_dataloader(dataset, batch_size=4, shuffle=False)

def eval_reconstruction(vae, loader, loss_fn, device):
    vae.eval()
    total_rec, total_chart, n = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            x = batch.to(device)
            z, A, c_spec, x_hat = vae(x, prior)
            losses = loss_fn(x, x_hat, A, A.detach())
            total_rec += losses['rec'].item()
            total_chart += losses['chart'].item()
            n += 1
    return {'rec': total_rec/n, 'chart': total_chart/n}

graph_metrics = eval_reconstruction(graph_vae, eval_loader, loss_fn, device)
baseline_metrics = eval_reconstruction(baseline_vae, eval_loader, loss_fn, device)

print('=== Reconstruction Comparison ===')
print(f'Graph decoder:    L1={graph_metrics["rec"]:.6f}  chart={graph_metrics["chart"]:.6f}')
print(f'Baseline decoder: L1={baseline_metrics["rec"]:.6f}  chart={baseline_metrics["chart"]:.6f}')
diff = baseline_metrics['rec'] - graph_metrics['rec']
print(f'Graph improvement: {diff:+.6f} (positive = graph is better)')

=== Reconstruction Comparison ===
Graph decoder:    L1=0.127685  chart=0.000000
Baseline decoder: L1=0.090009  chart=0.000000
Graph improvement: -0.037677 (positive = graph is better)


In [11]:
# lambda_ED ablation: graph decoder with vs without c_spec gating
class NoCSPecVAE(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x, prior):
        z, a, c_spec = self.encoder.encode(x, prior)
        zero_cspec = torch.zeros_like(c_spec)
        x_hat = self.decoder(z, zero_cspec)
        return z, a, c_spec, x_hat

ablation_vae = NoCSPecVAE(encoder, graph_decoder)
ablation_metrics = eval_reconstruction(ablation_vae, eval_loader, loss_fn, device)

print('=== lambda_ED Ablation ===')
print(f'With c_spec:    L1={graph_metrics["rec"]:.6f}  chart={graph_metrics["chart"]:.6f}')
print(f'Without c_spec: L1={ablation_metrics["rec"]:.6f}  chart={ablation_metrics["chart"]:.6f}')
diff = ablation_metrics['rec'] - graph_metrics['rec']
print(f'lambda_ED effect: {diff:+.6f} (positive = gating helps)')

=== lambda_ED Ablation ===
With c_spec:    L1=0.127685  chart=0.000000
Without c_spec: L1=0.127629  chart=0.000000
lambda_ED effect: -0.000057 (positive = gating helps)


## Summary

This notebook demonstrated the full ALD-SC sound generation pipeline:

- **Frozen EnCodec encoder** -> 1-D continuous latents z
- **ArrowSpace prior** (L_F, U_q, lambda_ED) from corpus features
- **Graph decoder** (project->gate->lift along U_q) vs **baseline decoder**
- **1-D DiT** unconditional diffusion on z
- **DDIM sampling** -> decode -> audio waveform

### Knobs recap
Re-run the generation cells with different `SEED`, `STEPS`,
`USE_C_SPEC`, `TEMPERATURE`, and `CLIP_INDEX` values to explore
the sound space.

### Next steps
- Train on real ESC-50 with EnCodecEncoder
- Add text/CLAP conditioning (Phase 2)
- Scale up DiT and decoder capacity
- Add the Barontini entropic clock to the decoder